# Retrieval from ChromaDB

Queries the ChromaDB collection built by `notebooks/ingestion_preprocessing_embedding.ipynb`.

**Run the ingestion notebook first** so `notebooks/chroma_db/` exists on disk.

In [1]:
!pip install chromadb transformers torch


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import chromadb
from transformers import AutoTokenizer, AutoModel
import torch

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_collection("actas_obra")
print(f"Collection loaded with {collection.count()} chunks.")

c:\REPOS\ML problems\POSTGRAU\RAG-UPCSchool-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection loaded with 55 chunks.


In [3]:
model_name = "BSC-LT/MrBERT-es"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 16786.24it/s]
[transformers] ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def embed_query(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().tolist()

In [7]:
search_query = "Se ha realizado el armado de la zapata del muro ?"

query_embedding = embed_query(search_query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"--- Rank {i+1} | Chunk ID: {meta['chunk_id']} | Distance: {dist:.4f} ---")
    print(doc)
    print()

--- Rank 1 | Chunk ID: 2 | Distance: 1243.4155 ---
12/25 Con fecha de 10 de diciembre de 2025, las partes se reúnen en el emplazamiento de obra situado en la estación de tren de Santa Perpètua de la Mogoda – La Florida con objeto de realizar la visita semanal de seguimiento de la obra. 1.2. Breve Reportaje fotográfico 04/12/25 Ejecución de armado de la zapata de muro de inicio de rampa. Está previsto hormigonar esta zapata el viernes 12 de diciembre 2025 Ejecución de hormigón de limpieza y armado de zapata de muro de andén tipo 3 de hormigón arm

--- Rank 2 | Chunk ID: 18 | Distance: 1263.1119 ---
5. Estabilidad del talud 5/12/25 UTE proporciona un plano de planta general “Planta general. Mediciones hincas.” En el documento gráfico de mediciones se indica: Se reportan tres alineaciones con longitudes: 1er carril: 219 m (73 uds × 3 m). 2º carril: 252 m (42 uds × 6 m). 3º carril: 252 m (42 uds × 6 m). Total carriles = 723 m. El Anejo de cálculo modela hincas de carril 54E1, y muestra fas